# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

ValueError: mount failed

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 31 04:36:46 AM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1961460,104.8,5673660,303.1,5673660,303.1
Vcells,16814887,128.3,193020336,1472.7,241207948,1840.3


### 2.2.2 Carga de Librerias

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- 59405
PARAM$semilla_primigenia <- 500029

In [ ]:
PARAM$kaggle$competencia <- "utn-2026-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [ ]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO
  #bagging_frec= 1.0,
  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 100,
  learning_rate= 0.10,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 32, # integer >= 2
  min_data_in_leaf= 20 # integer >= 0
)


### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83401

[1] 154

2.2.5 Configuracion del Grid Search

In [ ]:
# En el argumento x llegan los parmaetros de LightGBM
#  devuelve la AUC en cross validation del modelo entrenado

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # Inyectar las semillas específicas en la lista
  param_completo$seed <- x$seed
  param_completo$feature_fraction_seed <- x$seed
  param_completo$bagging_seed <- x$seed

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  return(AUC)
}

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_iterations= c(500),
  learning_rate= c(0.05, 0.1),
  num_leaves= c(100,200),
  min_data_in_leaf= c(100, 200),
  seed = c(300017, 500029, 700027, 900007,100103),
  #seed = c(500029),
  #bagging_freq = 1,
  #bagging_fraction = c(0.5),
  feature_fraction = c(0.5, 0.7)
)

In [ ]:
# veo que tiene la tabla antes de procesar
tb_nueva

num_iterations,learning_rate,num_leaves,min_data_in_leaf,seed,feature_fraction
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
500,0.05,100,100,100103,0.5
500,0.05,100,100,100103,0.7
500,0.05,100,100,300017,0.5
500,0.05,100,100,300017,0.7
500,0.05,100,100,500029,0.5
500,0.05,100,100,500029,0.7
500,0.05,100,100,700027,0.5
500,0.05,100,100,700027,0.7
500,0.05,100,100,900007,0.5


### Grid Search con checkpoint / reanudación

Cada combinación se guarda inmediatamente en Google Drive. Si Google Colab se corta, al volver a ejecutar el notebook se omiten las combinaciones ya guardadas y se continúa con las pendientes.


In [ ]:
# ============================================================
# GRID SEARCH CON CHECKPOINT / RESUME
# ============================================================

archivo_checkpoint <- "tb_grid_checkpoint.txt"

if (file.exists(archivo_checkpoint)) {

  tb_checkpoint <- fread(archivo_checkpoint)

  message(
    "Checkpoint encontrado: ",
    nrow(tb_checkpoint),
    " combinaciones ya procesadas."
  )

} else {

  tb_checkpoint <- data.table(
    num_iterations = integer(),
    learning_rate = numeric(),
    num_leaves = integer(),
    AUC = numeric()
  )

  message("No existe checkpoint. Comienzo desde cero.")
}


for (i in seq_len(nrow(tb_nueva))) {

  x <- list(
    num_iterations = tb_nueva[i, num_iterations],
    learning_rate = tb_nueva[i, learning_rate],
    num_leaves = tb_nueva[i, num_leaves]
  )

  ya_procesada <- nrow(
    tb_checkpoint[
      num_iterations == x$num_iterations &
      learning_rate == x$learning_rate &
      num_leaves == x$num_leaves
    ]
  ) > 0

  if (ya_procesada) {

    message(
      "SKIP [", i, "/", nrow(tb_nueva), "] ",
      "num_iterations=", x$num_iterations,
      " learning_rate=", x$learning_rate,
      " num_leaves=", x$num_leaves
    )

    next
  }


  message(
    "\nRUN [", i, "/", nrow(tb_nueva), "] ",
    "num_iterations=", x$num_iterations,
    " learning_rate=", x$learning_rate,
    " num_leaves=", x$num_leaves
  )

  AUC <- Estimar_AUC_lightgbm(x)


  resultado <- data.table(
    num_iterations = x$num_iterations,
    learning_rate = x$learning_rate,
    num_leaves = x$num_leaves,
    AUC = AUC
  )


  fwrite(
    resultado,
    file = archivo_checkpoint,
    sep = "\t",
    append = file.exists(archivo_checkpoint),
    col.names = !file.exists(archivo_checkpoint)
  )

  tb_checkpoint <- rbind(tb_checkpoint, resultado)

  message(
    "CHECKPOINT GUARDADO: AUC=", AUC
  )
}


tb_nueva <- merge(
  tb_nueva,
  tb_checkpoint,
  by = c(
    "num_iterations",
    "learning_rate",
    "num_leaves"
  ),
  all.x = TRUE
)

message(
  "\nGrid Search terminado. ",
  sum(!is.na(tb_nueva$AUC)),
  "/",
  nrow(tb_nueva),
  " combinaciones procesadas."
)

tb_nueva


No existe checkpoint. Comienzo desde cero.


RUN [1/80] num_iterations=500 learning_rate=0.05 num_leaves=100

Mon Aug 31 04:40:02 AM 2026  500, 0.05, 100 AUC 0.925966781378828

CHECKPOINT GUARDADO: AUC=0.925966781378828

SKIP [2/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [3/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [4/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [5/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [6/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [7/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [8/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [9/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [10/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [11/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [12/80] num_iterations=500 learning_rate=0.05 num_leaves=100

SKIP [13/80] num_iterations=500 learning_rate=0.05

num_iterations,learning_rate,num_leaves,min_data_in_leaf,seed,feature_fraction,AUC
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
500,0.05,100,100,100103,0.5,0.9259668
500,0.05,100,100,100103,0.7,0.9259668
500,0.05,100,100,300017,0.5,0.9259668
500,0.05,100,100,300017,0.7,0.9259668
500,0.05,100,100,500029,0.5,0.9259668
500,0.05,100,100,500029,0.7,0.9259668
500,0.05,100,100,700027,0.5,0.9259668
500,0.05,100,100,700027,0.7,0.9259668
500,0.05,100,100,900007,0.5,0.9259668


In [ ]:
# veo que tiene la tabla DESPUES de procesar
tb_nueva

num_iterations,learning_rate,num_leaves,min_data_in_leaf,seed,feature_fraction,AUC
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
500,0.05,100,100,100103,0.5,0.9259668
500,0.05,100,100,100103,0.7,0.9259668
500,0.05,100,100,300017,0.5,0.9259668
500,0.05,100,100,300017,0.7,0.9259668
500,0.05,100,100,500029,0.5,0.9259668
500,0.05,100,100,500029,0.7,0.9259668
500,0.05,100,100,700027,0.5,0.9259668
500,0.05,100,100,700027,0.7,0.9259668
500,0.05,100,100,900007,0.5,0.9259668


In [ ]:
# Guardo una copia completa y ordenada del resultado final del Grid Search
tb_grid_final <- copy(tb_nueva)
setorder(tb_grid_final, -AUC)

fwrite(
  tb_grid_final,
  file = "tb_grid_serach_01.txt",
  sep = "\t"
)


In [ ]:
# mejores hiperparametros
setorder(tb_nueva, -AUC)

if (nrow(tb_nueva) == 0 || all(is.na(tb_nueva$AUC))) {
  stop("No hay resultados validos del Grid Search.")
}

PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_nueva[1])

PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL

PARAM$out$lgbm$mejores_hiperparametros


$num_iterations
[1] 500

$learning_rate
[1] 0.05

$num_leaves
[1] 200

$min_data_in_leaf
[1] 100

$seed
[1] 100103

$feature_fraction
[1] 0.5

In [ ]:
write_yaml( PARAM, file="PARAM.yml")

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 100103

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 500

$learning_rate
[1] 0.05

$feature_fraction
[1] 0.5

$num_leaves
[1] 200

$min_data_in_leaf
[1] 100

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
# ============================================================
# FINAL TRAINING CON RESUME
# ============================================================

if (file.exists("modelo.txt")) {

  message("Modelo final encontrado en disco. Lo cargo y no vuelvo a entrenar.")

  modelo_final <- lgb.load("modelo.txt")

} else {

  message("No existe modelo final. Entrenando...")

  modelo_final <- lgb.train(
    data = dtrain,
    param = param_normalizado
  )

  # Guardar inmediatamente despues del entrenamiento.
  lgb.save(
    modelo_final,
    "modelo.txt"
  )

  message("Modelo final guardado en modelo.txt")
}


No existe modelo final. Entrenando...

Modelo final guardado en modelo.txt



In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(
  lgb.importance(modelo_final)
)

archivo_importancia <- "impo.txt"

fwrite(
  tb_importancia,
  file = archivo_importancia,
  sep = "\t"
)


In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [ ]:
# ============================================================
# SCORING CON RESUME
# ============================================================

archivo_prediccion <- "prediccion.txt"

if (file.exists(archivo_prediccion)) {

  message(
    "Prediccion encontrada en disco. La cargo y no vuelvo a calcularla."
  )

  tb_prediccion <- fread(archivo_prediccion)

} else {

  dfuture <- dataset[foto_mes == 202109]

  prediccion <- predict(
    modelo_final,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]

  fwrite(
    tb_prediccion,
    file = archivo_prediccion,
    sep = "\t"
  )

  message("Prediccion guardada en ", archivo_prediccion)
}


Prediccion guardada en prediccion.txt



Kaggle Competition Submit

In [ ]:
# genero archivos con los "envios" mejores
# suba TODOS los archivos a Kaggle

setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  archivo_ok <- paste0(
    "./kaggle/submit_",
    PARAM$experimento,
    "_",
    envios,
    ".ok"
  )

  if (file.exists(archivo_ok)) {
    message(
      "SKIP Kaggle envios=", envios,
      " - ya marcado como enviado."
    )
    next
  }


  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento,
    "_",
    envios,
    ".csv"
  )

  fwrite(
    tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)

  mensaje <- paste0(
    "-m 'envios=", envios,
    " semilla=", PARAM$semilla_primigenia,
    "'"
  )

  linea <- paste(comando, competencia, arch, mensaje)

  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")

  # Solo marco el envio como OK despues de ejecutar el submit.
  writeLines(
    c(
      paste("experimento:", PARAM$experimento),
      paste("envios:", envios),
      paste("fecha:", Sys.time())
    ),
    archivo_ok
  )

  Sys.sleep(45)
}


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 31 05:15:37 AM 2026"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.